# Heterogeneity Across Physical Stations
## The 15 SIMA stations individually: where is the morning association apparent?

The preceding analyses use six monitoring zones, each averaging physical stations from the same sector (notebook 00). Here we undo that aggregation and reconstruct the arrival-versus-mid-morning gap directly for all 15 physical stations using OBSERVED hours only. Zone-level imputation is not applied to individual stations.

This serves two purposes:

1. Examine whether the CO association is concentrated at particular stations or has a consistent sign across the network.
2. Prepare the station-level estimates for finer spatial comparisons, such as school density within each monitoring area. Those extensions require georeferenced school locations.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Diagnostic and sensitivity: the primary design disaggregated to the fifteen physical stations (Fig. 2 of the article). The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

output_dir = Path('../data_processed')
estaciones_dir = output_dir / 'estaciones_fisicas'
estaciones_dir.mkdir(exist_ok=True)

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'base_horaria_estaciones_fisicas.csv.gz',
                       output_dir / 'calendario_oficial_sep.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 2 files


## 2. Applying the design to ONE station

For physical station $s$ and date $t$:

$$ \text{gap}_{st}=\log\bar y_{st}^{7\text{–}9}-\log\bar y_{st}^{10\text{–}12}, $$

where the two means use available observed hours in the arrival and same-day comparison windows. A factor multiplying both windows equally at that station and date cancels in the difference of logarithms.

For EACH STATION separately, estimate

$$ \text{gap}_{st}=\alpha+\beta\text{School}_t+\gamma_{m(t)}+\delta_{d(t)}+\varepsilon_{st}, $$

with month and day-of-week fixed effects and ISO-week clustered errors, as elsewhere in the analysis. Report $(e^\beta-1)\times100$, transforming the endpoints of the coefficient's 95% confidence interval to percentages.

In [2]:
def contraste_gap(df, columna_y):
    "\n    Estimate the 'clase' coefficient for columna_y with month and weekday fixed effects\n    and ISO-week-clustered errors, as in the main analysis.\n    "
    df = df.dropna(subset=[columna_y]).copy()
    piezas = [pd.Series(1.0, index=df.index, name='const'), df['clase'].astype(float),
              pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float),
              pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float)]
    if df['estacion_fisica'].nunique() > 1:
        piezas.append(pd.get_dummies(df['estacion_fisica'], prefix='est', drop_first=True).astype(float))
    X = pd.concat(piezas, axis=1)
    Xv = X.values
    n, k = Xv.shape
    if n <= k or df['clase'].nunique() < 2:
        return None
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    y = df[columna_y].values
    beta_todos = XtX_inv @ Xv.T @ y
    residuos = y - Xv @ beta_todos

    grupos = pd.factorize(df['wk'])[0]
    orden = np.argsort(grupos, kind='stable')
    Xo, ro, go = Xv[orden], residuos[orden], grupos[orden]
    inicios = np.flatnonzero(np.r_[True, go[1:] != go[:-1]])
    G = len(inicios)
    S = np.zeros((k, k))
    fines = np.r_[inicios[1:], len(go)]
    for s, e in zip(inicios, fines):
        zg = Xo[s:e].T @ ro[s:e]
        S += np.outer(zg, zg)
    V = XtX_inv @ S @ XtX_inv
    V *= (G / (G - 1)) * ((n - 1) / (n - k))

    idx_clase = X.columns.get_loc('clase')
    beta = beta_todos[idx_clase]
    se = np.sqrt(V[idx_clase, idx_clase])
    z = beta / se
    return {'beta': beta, 'se': se,
            'pct': (np.exp(beta) - 1) * 100,
            'ci_lo_pct': (np.exp(beta - 1.96 * se) - 1) * 100,
            'ci_hi_pct': (np.exp(beta + 1.96 * se) - 1) * 100,
            'p': 2 * (1 - stats.norm.cdf(abs(z))),
            'n': n, 'G_semanas': G}

print('Contrast function ready')

Contrast function ready


## 3. Building the physical-station gap panel

Use OBSERVED hours only. A station–date cell enters if at least one valid hour is available in each window. Record the number of contributing dates for each station.

In [3]:
base = pd.read_csv('../data_processed/base_horaria_estaciones_fisicas.csv.gz', parse_dates=['date'])
calendario = pd.read_csv('../data_processed/calendario_oficial_sep.csv', parse_dates=['date'])
clase_por_dia = calendario.set_index('date')['lectivo'].astype(int)

base['fecha'] = base['date'].dt.normalize()
base['hora'] = base['date'].dt.hour
base['franja'] = np.where(base['hora'].isin([7, 8]), 'pico',
                          np.where(base['hora'].isin([10, 11]), 'control', 'otra'))

franjas = base[base['franja'] != 'otra']
medias = (franjas.pivot_table(index=['estacion_fisica', 'zona', 'fecha', 'parametro'],
                              columns='franja', values='valor', aggfunc='mean')
          .dropna(subset=['pico', 'control']).reset_index())
medias = medias[(medias['pico'] > 0) & (medias['control'] > 0)]
medias['gap'] = np.log(medias['pico']) - np.log(medias['control'])

panel = medias.pivot_table(index=['estacion_fisica', 'zona', 'fecha'],
                           columns='parametro', values='gap').reset_index()
panel = panel.rename(columns={c: f'gap_{c}' for c in contaminantes})
panel['clase'] = panel['fecha'].map(clase_por_dia)
panel = panel.dropna(subset=['clase'])
panel['clase'] = panel['clase'].astype(int)
panel['mes'] = panel['fecha'].dt.month
panel['dow'] = panel['fecha'].dt.dayofweek
panel['wk'] = (panel['fecha'].dt.isocalendar().year.astype(int) * 100
               + panel['fecha'].dt.isocalendar().week.astype(int))

print("=" * 70)
print('PHYSICAL-STATION GAP PANEL (observed hours only)')
print("=" * 70)
dias_por_estacion = panel.groupby('estacion_fisica')['fecha'].nunique().sort_values()
print(f"Rows: {len(panel):,} | stations: {panel['estacion_fisica'].nunique()}")
print('\nDays with a CO gap by station:')
print(panel.dropna(subset=['gap_CO']).groupby('estacion_fisica')['fecha'].nunique().to_string())

PHYSICAL-STATION GAP PANEL (observed hours only)
Rows: 16,119 | stations: 15

Days with a CO gap by station:
estacion_fisica
CENTRO        1011
NORESTE       1031
NORESTE 3      988
NORESTE2       910
NOROESTE       930
NOROESTE 2    1070
NOROESTE 3     889
NORTE         1065
NORTE 2       1076
SUR           1021
SURESTE        949
SURESTE 3      995
SURESTE2       932
SUROESTE      1069
SUROESTE2     1041


## 4. Station-specific gap models for CO and NO2

In [4]:
resultados_filas = []
for estacion_f, sub in panel.groupby('estacion_fisica'):
    zona = sub['zona'].iloc[0]
    for c in ['CO', 'NO2']:
        r = contraste_gap(sub, f'gap_{c}')
        if r is None:
            continue
        r.update({'estacion_fisica': estacion_f, 'zona': zona, 'contaminante': c})
        resultados_filas.append(r)

tabla_estaciones = pd.DataFrame(resultados_filas)[
    ['contaminante', 'zona', 'estacion_fisica', 'pct', 'ci_lo_pct', 'ci_hi_pct', 'p', 'n', 'G_semanas']]
for col in ['pct', 'ci_lo_pct', 'ci_hi_pct']:
    tabla_estaciones[col] = tabla_estaciones[col].round(2)
tabla_estaciones['p'] = tabla_estaciones['p'].round(4)
tabla_estaciones = tabla_estaciones.sort_values(['contaminante', 'zona', 'estacion_fisica'])

print("=" * 70)
print('PHYSICAL-STATION GAP DiD (school-term effect on the 7-9 vs 10-12 ratio, %)')
print("=" * 70)
print(tabla_estaciones.to_string(index=False))

tabla_estaciones.to_csv(estaciones_dir / 'gap_did_por_estacion_fisica.csv', index=False)
print(f"\nSaved to: {estaciones_dir / 'gap_did_por_estacion_fisica.csv'}")

PHYSICAL-STATION GAP DiD (school-term effect on the 7-9 vs 10-12 ratio, %)
contaminante      zona estacion_fisica   pct  ci_lo_pct  ci_hi_pct      p    n  G_semanas
          CO    CENTRO          CENTRO  1.13      -2.09       4.45 0.4971 1011        157
          CO  NORESTE3         NORESTE  5.63       2.20       9.17 0.0011 1031        155
          CO  NORESTE3       NORESTE 3  0.15      -4.61       5.15 0.9517  988        147
          CO  NORESTE3        NORESTE2  8.34      -0.54      18.00 0.0663  910        141
          CO NOROESTE3        NOROESTE  1.50      -3.74       7.02 0.5820  930        143
          CO NOROESTE3      NOROESTE 2  4.65      -1.40      11.08 0.1349 1070        158
          CO NOROESTE3      NOROESTE 3  5.97      -1.16      13.61 0.1029  889        136
          CO    NORTE2           NORTE  9.46       1.19      18.40 0.0241 1065        157
          CO    NORTE2         NORTE 2  5.56       1.14      10.18 0.0132 1076        158
          CO       SUR   

## 5. Consistency: pooling all 15 stations versus the zone-level result

Pool the 15 stations in one regression with physical-station, month and day-of-week fixed effects. Comparing this estimate with notebook 05's six-zone estimate examines sensitivity to aggregation and weighting, using the same conceptual design without averaging stations first.

In [5]:
print("=" * 70)
print('ALL 15 PHYSICAL STATIONS (physical-station fixed effects)')
print("=" * 70)
conjunto = {}
for c in contaminantes:
    r = contraste_gap(panel, f'gap_{c}')
    conjunto[c] = r
    marca = ' *' if r['p'] < 0.05 else ''
    print(f"  {c}: {r['pct']:+.2f}% [{r['ci_lo_pct']:.2f}, {r['ci_hi_pct']:.2f}] "
          f"p = {r['p']:.4f}{marca}  (n = {r['n']:,})")

pd.DataFrame(conjunto).T.rename_axis('contaminante').to_csv(estaciones_dir / 'gap_did_15_estaciones_juntas.csv')
print(f"\nSaved to: {estaciones_dir / 'gap_did_15_estaciones_juntas.csv'}")

ALL 15 PHYSICAL STATIONS (physical-station fixed effects)
  CO: +4.66% [1.07, 8.38] p = 0.0105 *  (n = 14,977)
  NO2: +1.16% [-6.46, 9.39] p = 0.7734  (n = 15,313)
  O3: -3.76% [-14.03, 7.75] p = 0.5063  (n = 15,275)
  PM10: -0.85% [-5.45, 3.97] p = 0.7248  (n = 15,826)
  PM2.5: +2.77% [-2.77, 8.63] p = 0.3342  (n = 11,757)

Saved to: ../data_processed/estaciones_fisicas/gap_did_15_estaciones_juntas.csv


## 6. Station-level CO plot, grouped by zone

Station names below retain their names in the original SIMA workbooks. The paper's Table 1 and Figure 2 use the network's sector codes:

CE = CENTRO; NE = NORESTE, NE2 = NORESTE2, NE3 = NORESTE 3; NO = NOROESTE, NO2 = NOROESTE 2, NO3 = NOROESTE 3; NTE = NORTE, NTE2 = NORTE 2; SE = SURESTE, SE2 = SURESTE2, SE3 = SURESTE 3, SUR = SUR; SO = SUROESTE, SO2 = SUROESTE2.

Figure 2 reports these same CO estimates with 95% intervals. Filled markers indicate intervals excluding zero. The dashed line and band show the pooled 15-station estimate from Section 5. Grouping stations by sector codes makes the figure comparable with Table 1. Provider names and codes are preserved as proper identifiers.

In [6]:
colores_zona = {'CENTRO': '#8e44ad', 'NORESTE3': '#e67e22', 'NOROESTE3': '#16a085',
                'NORTE2': '#c0392b', 'SUR': '#2980b9', 'SUROESTE2': '#27ae60'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
for ax, c in zip(axes, ['CO', 'NO2']):
    sub = tabla_estaciones[tabla_estaciones['contaminante'] == c].reset_index(drop=True)
    y_pos = np.arange(len(sub))
    for i, fila in sub.iterrows():
        color = colores_zona[fila['zona']]
        ax.errorbar(fila['pct'], i, xerr=[[fila['pct'] - fila['ci_lo_pct']],
                                          [fila['ci_hi_pct'] - fila['pct']]],
                    fmt='o', color=color, capsize=4, markersize=7)
    ax.axvline(0, color='black', linewidth=1)
    ax.axvline(conjunto[c]['pct'], color='gray', linewidth=1.5, linestyle='--',
               label=f"all 15 pooled: {conjunto[c]['pct']:+.1f}%")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['estacion_fisica'] + '  (' + sub['zona'] + ')', fontsize=9)
    ax.set_title(c, fontsize=13, fontweight='bold')
    ax.set_xlabel('School-term effect on the gap (%)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)

fig.suptitle('Gap DiD by SIMA physical station (transformed 95% CI; weekly-clustered errors)\nColors indicate main-analysis zones',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(estaciones_dir / '01_gap_did_por_estacion_fisica.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {estaciones_dir / '01_gap_did_por_estacion_fisica.png'}")

Plot saved to: ../data_processed/estaciones_fisicas/01_gap_did_por_estacion_fisica.png


## 7. Interpretation and remaining spatial extensions

In [7]:
co = tabla_estaciones[tabla_estaciones['contaminante'] == 'CO']
positivos = int((co['pct'] > 0).sum())
significativos = co[co['p'] < 0.05]['estacion_fisica'].tolist()

print("=" * 70)
print("LECTURA")
print("=" * 70)
import json
co_zonas = json.load(open('../data_processed/resultados_metodos/did_franjas.json'))['hibrido']['did_franjas']['por_var']['gap_CO']
print(f"1. Pooling all 15 stations, the CO gap effect is {conjunto['CO']['pct']:+.2f}% "
      f"(p = {conjunto['CO']['p']:.4f}):")
print(f"   compared with {co_zonas['pct']:+.1f}% for the six zones in notebook 05; disaggregation")
print('   changes the spatial resolution of the analysis.')
print(f"2. Station by station, {positivos} of {len(co)} stations have positive CO gap estimates;")
print(f"   individually significant:{significativos if significativos else 'none'}")
print('   (each station has fewer observations; wider CIs are expected).')
print('3. POSSIBLE EXTENSION: classify stations by nearby school density,')
print('   using a justified spatial buffer, and compare high- and low-density groups.')
print('   This requires georeferenced schools; the physical-station model in this')
print('   notebook could be combined with that information.')

print('\nCOMPLETED')

LECTURA
1. Pooling all 15 stations, the CO gap effect is +4.66% (p = 0.0105):
   compared with +4.6% for the six zones in notebook 05; disaggregation
   changes the spatial resolution of the analysis.
2. Station by station, 15 of 15 stations have positive CO gap estimates;
   individually significant:['NORESTE', 'NORTE', 'NORTE 2', 'SUROESTE']
   (each station has fewer observations; wider CIs are expected).
3. POSSIBLE EXTENSION: classify stations by nearby school density,
   using a justified spatial buffer, and compare high- and low-density groups.
   This requires georeferenced schools; the physical-station model in this
   notebook could be combined with that information.

COMPLETED
